# Prompt construction check

This notebook imports the live construction helpers from `predict.py`, inspects the selected rule blocks, and reproduces the prompt assembly performed before `_ask_llm()` calls the model. It does not make an API call.

In [ ]:
from pathlib import Path
import json
import sys
import re

# Jupyter normally starts this notebook inside notebooks/. Add the repo root
# so the top-level predict.py module can be imported.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from predict import (
    GLOBAL_PATH,
    INDUSTRY_PATH,
    PROMPT_PATH,
    load_prompt_rules,
    load_yaml,
)

## 1. Confirm the runtime files resolve

In [ ]:
runtime_paths = {
    "prompt": PROMPT_PATH,
    "global_playbook": GLOBAL_PATH,
    "industry_playbooks": INDUSTRY_PATH,
}

for name, path in runtime_paths.items():
    print(f"{name:20} exists={path.exists()}  path={path}")
    assert path.exists(), f"Missing runtime file: {path}"

## 2. Inspect available industries and choose one

In [ ]:
global_playbook = load_yaml(GLOBAL_PATH)
industry_playbooks = load_yaml(INDUSTRY_PATH)

reserved_keys = {"meta", "quarter_calibration"}
available_industries = [
    key for key, value in industry_playbooks.items()
    if key not in reserved_keys and isinstance(value, dict)
]

print("Global principles:", len(global_playbook.get("principles", [])))
print("Global rules:", len(global_playbook.get("rules", [])))
print("Quarter calibration rules:", len(industry_playbooks.get("quarter_calibration", [])))
print("Industries:", available_industries)

INDUSTRY = "BusEq"
assert INDUSTRY in available_industries

## 3. Load exactly the blocks used by the prompt

In [ ]:
core_directive, industry_rules = load_prompt_rules(INDUSTRY)

print("CORE DIRECTIVE\n")
print(core_directive)
print("\nAPPLICABLE RULES (first 4,000 characters)\n")
print(industry_rules[:4000])

assert "Q3-CAL-01" in industry_rules, "Quarter calibration must apply to every event"
assert industry_playbooks[INDUSTRY]["rules"][0]["id"] in industry_rules

other_industry = next(name for name in available_industries if name != INDUSTRY)
other_rule_ids = [rule["id"] for rule in industry_playbooks[other_industry].get("rules", [])]
if other_rule_ids:
    assert other_rule_ids[0] not in industry_rules, f"Rules from {other_industry} leaked into the prompt"

## 4. Reproduce `_ask_llm()` prompt construction

Edit the sample values below to exercise other events. The dossier is a string for now; replace it with serialized dossier YAML when that step is implemented.

In [ ]:
sample_summary = {
    "summary": [
        "Revenue exceeded consensus expectations.",
        "Management raised full-year guidance.",
        "Capital expenditure guidance also increased.",
    ]
}
ticker = "TEST"
event_type = "EARNINGS_RELEASE"
dossier = "No cached dossier is available."

summary_value = sample_summary.get("summary")
if isinstance(summary_value, list):
    summary_text = "\n".join(f"- {bullet}" for bullet in summary_value)
elif summary_value:
    summary_text = str(summary_value)
else:
    summary_text = json.dumps(sample_summary, ensure_ascii=False)
summary_text = summary_text[:8000]

prompt_template = PROMPT_PATH.read_text(encoding="utf-8")
prompt_template = re.sub(
    r"\A\s*<!--.*?-->\s*",
    "",
    prompt_template,
    count=1,
    flags=re.DOTALL,
)

user_prompt = (
    prompt_template
    .replace("{event_bullets}", summary_text)
    .replace("{core_directive}", core_directive)
    .replace("{industry_rules}", industry_rules)
    .replace("{dossier}", dossier)
)

print(user_prompt)

## 5. Validate the finished prompt

In [ ]:
expected_placeholders = [
    "{event_bullets}",
    "{core_directive}",
    "{industry_rules}",
    "{dossier}",
]
unresolved = [item for item in expected_placeholders if item in user_prompt]

assert not unresolved, f"Unresolved prompt placeholders: {unresolved}"
assert summary_text in user_prompt
assert "Q3-CAL-01" in user_prompt
assert industry_playbooks[INDUSTRY]["rules"][0]["id"] in user_prompt

print(f"Prompt construction passed for {ticker} / {INDUSTRY}.")
print(f"Final prompt length: {len(user_prompt):,} characters")